
## Notebook 04: Combined Full Moments Factor Inventory
**Input:** `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_combined_full_moments.parquet`
**Inventories:** all four Stage 1.5 inventories as above

### Logic
Same schema-read approach. Features are classified in two stages:

1. **Monthly vs daily:** detected by the `monthly_` prefix. The prefix is stripped before further processing.
2. **Moment vs macro:** within each frequency, a moment suffix (`_cwmean`, `_cwstd`, `_cwskew`, `_cwkurt`, `_spread`) is searched for. If found, the base factor name is looked up in the appropriate stock inventory (daily or monthly). If no suffix matches, the column is treated as a raw macro level and looked up in the corresponding macro inventory (daily or monthly).

For monthly stock moment columns the full column name has the structure `monthly_{base_factor}_{moment_type}`. The `monthly_` prefix is stripped first, then the moment suffix is stripped from the remainder to recover `base_factor`.

The `stock_skew_chg_5d` rename conflict is handled for daily stock moment columns.

Unmatched columns are flagged. Breakdowns by frequency, panel, and moment type are printed.

### Output Columns
`column`, `base_factor`, `moment_type` (cwmean / cwstd / cwskew / cwkurt / spread / raw level), `frequency` (daily / monthly), `panel` (A / B / C / D), `source`, `category`, `description`

**Output:** `Data/Data_Collection/Final/Stage_3_Model_Ready/combined_full_moments_factor_inventory.csv`

---

## Key Notes
- Neither notebook loads the full parquet data -- only the schema is read via `pyarrow.parquet.read_schema`.
- The `monthly_` prefix is the sole mechanism distinguishing daily from monthly features in the combined tables. It is applied to all monthly columns (both stock moments and macro levels) during the Stage 3 merge.
- All four Stage 1.5 inventories are loaded in both notebooks, but for the means table only cwmean and raw level lookups are needed; the full suffix-stripping logic only applies to the full moments table.
- The `stock_skew_chg_5d` rename (from the Stage 2 daily merge conflict) only affects daily stock features and is not relevant for monthly features.
- Unmatched columns would indicate a naming discrepancy between the Stage 3 combined tables and the Stage 1.5 inventories, requiring investigation.

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Load feature names from combined full moments
BASE = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')
schema = pq.read_schema(BASE / 'model_market_combined_full_moments.parquet')
all_cols = [f.name for f in schema]
features = [c for c in all_cols if c not in ['date', 'target_daily_return', 'target_monthly_return']]

# Load all four inventories
INV_DIR = Path('../../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
stock_daily_inv = pd.read_csv(INV_DIR / 'stock_daily_factor_inventory_final.csv')
macro_daily_inv = pd.read_csv(INV_DIR / 'macro_daily_factor_inventory_final.csv')
stock_monthly_inv = pd.read_csv(INV_DIR / 'stock_monthly_factor_inventory_final.csv')
macro_monthly_inv = pd.read_csv(INV_DIR / 'macro_monthly_factor_inventory_final.csv')

stock_daily_map = stock_daily_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')
macro_daily_map = macro_daily_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')
stock_monthly_map = stock_monthly_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')
macro_monthly_map = macro_monthly_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')

# Handle renamed conflict
stock_daily_map['stock_skew_chg_5d'] = stock_daily_map.get('skew_chg_5d', {})

moment_suffixes = ['_cwmean', '_cwstd', '_cwskew', '_cwkurt', '_spread']

rows = []
matched = 0
unmatched = []

for f in features:
    if f.startswith('monthly_'):
        # Monthly feature — strip prefix first
        without_prefix = f[len('monthly_'):]
        
        # Check if it's a moment column (stock monthly with suffix)
        is_moment = False
        for suffix in moment_suffixes:
            if without_prefix.endswith(suffix):
                base_name = without_prefix[:-len(suffix)]
                moment_type = suffix[1:]
                
                if base_name in stock_monthly_map:
                    info = stock_monthly_map[base_name]
                    rows.append({
                        'column': f,
                        'base_factor': base_name,
                        'moment_type': moment_type,
                        'frequency': 'monthly',
                        'panel': 'B (stock monthly)',
                        'source': info.get('source', ''),
                        'category': info.get('category', ''),
                        'description': f"{moment_type} of: {info.get('description', '')}",
                    })
                    matched += 1
                    is_moment = True
                    break
        
        if is_moment:
            continue
        
        # Not a moment — raw monthly macro
        if without_prefix in macro_monthly_map:
            info = macro_monthly_map[without_prefix]
            rows.append({
                'column': f,
                'base_factor': without_prefix,
                'moment_type': 'raw level',
                'frequency': 'monthly',
                'panel': 'D (macro monthly)',
                'source': info.get('source', ''),
                'category': info.get('category', ''),
                'description': info.get('description', ''),
            })
            matched += 1
        else:
            rows.append({
                'column': f, 'base_factor': without_prefix, 'moment_type': '???',
                'frequency': 'monthly', 'panel': '???', 'source': '', 'category': '', 'description': '',
            })
            unmatched.append(f)
    
    else:
        # Daily feature — check if it's a moment column (stock daily with suffix)
        is_moment = False
        for suffix in moment_suffixes:
            if f.endswith(suffix):
                base_name = f[:-len(suffix)]
                moment_type = suffix[1:]
                
                if base_name == 'stock_skew_chg_5d':
                    lookup = 'skew_chg_5d'
                else:
                    lookup = base_name
                
                if lookup in stock_daily_map:
                    info = stock_daily_map[lookup]
                    rows.append({
                        'column': f,
                        'base_factor': base_name,
                        'moment_type': moment_type,
                        'frequency': 'daily',
                        'panel': 'A (stock daily)',
                        'source': info.get('source', ''),
                        'category': info.get('category', ''),
                        'description': f"{moment_type} of: {info.get('description', '')}",
                    })
                    matched += 1
                    is_moment = True
                    break
        
        if is_moment:
            continue
        
        # Not a moment — raw daily macro
        if f in macro_daily_map:
            info = macro_daily_map[f]
            rows.append({
                'column': f,
                'base_factor': f,
                'moment_type': 'raw level',
                'frequency': 'daily',
                'panel': 'C (macro daily)',
                'source': info.get('source', ''),
                'category': info.get('category', ''),
                'description': info.get('description', ''),
            })
            matched += 1
        else:
            rows.append({
                'column': f, 'base_factor': f, 'moment_type': '???',
                'frequency': 'daily', 'panel': '???', 'source': '', 'category': '', 'description': '',
            })
            unmatched.append(f)

result = pd.DataFrame(rows)

print(f"Total features: {len(features)}")
print(f"Matched: {matched}")
print(f"Unmatched: {len(unmatched)}")
if unmatched:
    print(f"\nUnmatched columns:")
    for c in unmatched:
        print(f"  {c}")

print(f"\nBy frequency:")
print(result['frequency'].value_counts().to_string())

print(f"\nBy panel:")
print(result['panel'].value_counts().to_string())

print(f"\nBy moment type:")
print(result['moment_type'].value_counts().to_string())

# Save
out_path = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready/combined_full_moments_factor_inventory.csv')
result.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"  {len(result)} rows")

Total features: 2212
Matched: 2212
Unmatched: 0

By frequency:
frequency
daily      1145
monthly    1067

By panel:
panel
A (stock daily)      936
B (stock monthly)    934
C (macro daily)      209
D (macro monthly)    133

By moment type:
moment_type
cwmean       380
cwstd        380
cwskew       371
cwkurt       371
spread       368
raw level    342

Saved: ..\..\..\..\Data\Data_Collection\Final\Stage_3_Model_Ready\combined_full_moments_factor_inventory.csv
  2212 rows
